# 第18课：大模型微调（Fine-tuning）

## 学习目标
- 理解为什么需要微调，微调解决了什么问题
- 掌握 Full Fine-tuning vs Parameter-Efficient Fine-tuning (PEFT) 的核心区别
- 理解 LoRA 的原理：为什么只调很少参数就能生效
- 通过代码直觉理解微调流程

## 核心概念

上一课我们学了 BERT 和 GPT——预训练语言模型。它们在海量数据上学会了「语言的规则」，但不会直接做你的具体任务。

**微调 = 让通才变专才。**

类比：预训练模型就像一个大学毕业生，什么都懂一点；微调就是入职培训，让他快速适应你的岗位。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

## 1. 为什么需要微调？

预训练模型学到了通用语言知识，但：
- 你的任务可能有特殊格式（医疗报告、法律文书、代码）
- 你的领域有专业术语通用模型不熟悉
- 你需要特定的输出风格或格式

三种使用大模型的方式：
| 方式 | 改不改参数 | 成本 | 效果 |
|------|-----------|------|------|
| Prompt Engineering | ❌ 不改 | 极低 | 有限 |
| Fine-tuning | ✅ 改 | 中-高 | 显著提升 |
| 从零训练 | ✅ 全部 | 极高 | 最灵活 |

In [ ]:
# 可视化：三种方式的参数量 vs 效果对比
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左图：可训练参数量对比
methods = ['Prompt\nEngineering', 'LoRA\n(PEFT)', 'Full\nFine-tuning', 'From\nScratch']
trainable_params = [0, 0.5, 100, 100]  # 相对单位
colors = ['#4CAF50', '#FF9800', '#F44336', '#9C27B0']

bars = axes[0].bar(methods, trainable_params, color=colors, alpha=0.8, edgecolor='white', linewidth=2)
axes[0].set_ylabel('可训练参数量（相对单位）', fontsize=12)
axes[0].set_title('可训练参数量对比', fontsize=14, fontweight='bold')
axes[0].set_ylim(0, 120)
for bar, val in zip(bars, trainable_params):
    if val > 0:
        axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 2, 
                     f'{val}%', ha='center', fontsize=11, fontweight='bold')

# 右图：效果雷达图简化版 - 用柱状图代替
categories = ['领域适配', '成本效率', '数据需求', '部署难度']
prompt_scores = [2, 5, 5, 5]
lora_scores = [4, 4, 4, 4]
full_scores = [5, 1, 2, 2]

x = np.arange(len(categories))
width = 0.25
axes[1].bar(x - width, prompt_scores, width, label='Prompt Eng', color='#4CAF50', alpha=0.8)
axes[1].bar(x, lora_scores, width, label='LoRA (PEFT)', color='#FF9800', alpha=0.8)
axes[1].bar(x + width, full_scores, width, label='Full Fine-tune', color='#F44336', alpha=0.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels(categories)
axes[1].set_ylabel('得分 (1-5)', fontsize=12)
axes[1].set_title('三种方式综合对比', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].set_ylim(0, 6)

plt.tight_layout()
plt.savefig('docs/18_comparison.png', dpi=100, bbox_inches='tight')
plt.show()
print('三种方式各有适用场景，LoRA 是当前工业界最流行的折中方案')

## 2. LoRA 的核心直觉

LoRA（Low-Rank Adaptation）的核心思想：

**不需要改全部参数，用两个小矩阵的乘积来近似权重的更新量。**

原权重矩阵 W（d×d）的更新：
```
W' = W + ΔW
ΔW = A × B    （A: d×r, B: r×d, r << d）
```

直觉：就像装修房子，不需要拆掉重建（全量微调），只需要贴几块装饰板（低秩矩阵）就能改变风格。

| 概念 | Full Fine-tuning | LoRA |
|------|-----------------|------|
| 训练参数 | 全部（如 7B） | 极少（如 4M ≈ 0.06%） |
| 显存需求 | 极高 | 显著降低 |
| 推理方式 | 直接用 | 合并回原权重后相同 |
| 多任务 | 需要多个完整模型 | 只需切换小矩阵 |

In [ ]:
# LoRA 直觉演示：低秩近似的效果
np.random.seed(42)

# 模拟一个权重更新矩阵（真实场景中这就是梯度更新）
d = 100  # 原始维度
r = 4    # LoRA 秩（非常小！）

# 生成一个「真实的」更新矩阵
true_update = np.random.randn(d, d) * 0.1

# LoRA 近似：ΔW = A @ B
A = np.random.randn(d, r) * 0.1
B = np.random.randn(r, d) * 0.1
lora_update = A @ B

print(f'原权重矩阵参数量: {d*d:,}')
print(f'LoRA 参数量: {d*r + r*d:,}')
print(f'参数压缩比: {d*d / (d*r + r*d):.1f}x')
print(f'\n对于 GPT-2 (768维), r=8:')
print(f'  原参数: {768*768:,}')
print(f'  LoRA: {768*8*2:,}')
print(f'  压缩比: {768*768 / (768*8*2):.0f}x')

# 可视化低秩近似
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

im0 = axes[0].imshow(true_update[:20,:20], cmap='RdBu_r', vmin=-0.3, vmax=0.3)
axes[0].set_title('Full Update (ΔW)\n100×100 = 10,000 params', fontsize=11)
plt.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(A[:20,:], cmap='RdBu_r', vmin=-0.3, vmax=0.3)
axes[1].set_title(f'Matrix A (d×r)\n100×{r} = {d*r} params', fontsize=11)
plt.colorbar(im1, ax=axes[1], fraction=0.046)

im2 = axes[2].imshow(lora_update[:20,:20], cmap='RdBu_r', vmin=-0.3, vmax=0.3)
axes[2].set_title(f'LoRA Approx (A×B)\nTotal: {d*r + r*d} params', fontsize=11)
plt.colorbar(im2, ax=axes[2], fraction=0.046)

plt.suptitle('LoRA: 用极少的参数近似权重更新', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('docs/18_lora_demo.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# 模拟微调训练过程：Full vs LoRA 的学习曲线对比
np.random.seed(42)
steps = np.arange(0, 100)

# 模拟损失曲线
full_loss = 2.5 * np.exp(-0.03 * steps) + 0.3 + np.random.randn(100) * 0.05
lora_loss = 2.5 * np.exp(-0.025 * steps) + 0.35 + np.random.randn(100) * 0.05
prompt_loss = 2.5 * np.exp(-0.01 * steps) + 0.8 + np.random.randn(100) * 0.05

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 左：损失曲线
ax1.plot(steps, prompt_loss, label='Prompt Engineering', color='#4CAF50', alpha=0.8)
ax1.plot(steps, lora_loss, label='LoRA Fine-tuning', color='#FF9800', alpha=0.8, linewidth=2)
ax1.plot(steps, full_loss, label='Full Fine-tuning', color='#F44336', alpha=0.8)
ax1.set_xlabel('Training Steps')
ax1.set_ylabel('Loss')
ax1.set_title('训练损失曲线对比', fontsize=14, fontweight='bold')
ax1.legend()
ax1.set_ylim(0, 3)

# 右：显存占用对比
model_sizes = ['GPT-2\n(124M)', 'Llama-7B', 'Llama-13B', 'Llama-70B']
full_mem = [0.5, 28, 52, 280]  # GB (approximate)
lora_mem = [0.3, 16, 28, 140]

x = np.arange(len(model_sizes))
width = 0.35
ax2.bar(x - width/2, full_mem, width, label='Full Fine-tune', color='#F44336', alpha=0.8)
ax2.bar(x + width/2, lora_mem, width, label='LoRA', color='#FF9800', alpha=0.8)
ax2.set_xticks(x)
ax2.set_xticklabels(model_sizes)
ax2.set_ylabel('GPU 显存需求 (GB)')
ax2.set_title('显存占用对比', fontsize=14, fontweight='bold')
ax2.legend()

plt.tight_layout()
plt.savefig('docs/18_training_comparison.png', dpi=100, bbox_inches='tight')
plt.show()
print('关键发现：LoRA 以极少的参数量，达到接近全量微调的效果，同时大幅降低显存需求')

## 总结

| 要点 | 内容 |
|------|------|
| 微调本质 | 让预训练模型适配特定任务/领域，不改架构只调参数 |
| Full Fine-tuning | 改全部参数，效果好但成本极高 |
| LoRA | 用低秩矩阵近似更新，只调 0.1% 参数就能接近全量效果 |
| QLoRA | LoRA + 4bit 量化，进一步降低显存，单卡可微调 7B 模型 |
| 工程选择 | 小数据/资源有限 → LoRA；大数据/追求极致 → Full FT |

## 课后思考
1. 如果你有 1000 条标注数据和一个 7B 模型，会选择 Full FT 还是 LoRA？为什么？
2. LoRA 的秩 r 设多大合适？r 越大越好吗？
3. 为什么 LoRA 训练完后可以「合并回原权重」，推理时零额外开销？